In [4]:
# extract

import requests
import os 
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
LAT     = os.getenv("LAT")
LON     = os.getenv("LON")

def extract_weather(LAT,LON,API_KEY):
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={LAT}&lon={LON}&appid={API_KEY}"
    
    response = requests.get(url)

    response.raise_for_status()

    return  response.json()

data = extract_weather(LAT,LON,API_KEY)

print(data)

    

{'coord': {'lon': 10.99, 'lat': 44.34}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03d'}], 'base': 'stations', 'main': {'temp': 288.5, 'feels_like': 288.1, 'temp_min': 286.8, 'temp_max': 288.51, 'pressure': 1013, 'humidity': 77, 'sea_level': 1013, 'grnd_level': 946}, 'visibility': 10000, 'wind': {'speed': 1.02, 'deg': 35, 'gust': 1.06}, 'clouds': {'all': 40}, 'dt': 1776679167, 'sys': {'type': 2, 'id': 2004688, 'country': 'IT', 'sunrise': 1776659049, 'sunset': 1776708326}, 'timezone': 7200, 'id': 3163858, 'name': 'Zocca', 'cod': 200}


In [3]:
# transform

import pandas as pd
from extract import data


def transform_weather(data):

   filtered_data = [{
        "City"        : data.get("name"),
        "Country"     : data.get("sys", {}).get("country"),
        "Condition"   : data.get("weather", [{}])[0].get("description"),
        "Temp_K"      : data.get("main", {}).get("temp"),
        "Feels_Like_K": data.get("main", {}).get("feels_like"),
        "Humidity"    : data.get("main", {}).get("humidity")
   }]

   df = pd.DataFrame(filtered_data)

   df["Temp_C"] = (df["Temp_K"] - 273.15).round(2)

   df["Feels_Like_C"] = (df["Feels_Like_K"] - 273.15).round(2)

   df = df.drop(columns=["Temp_K", "Feels_Like_K"])

   df["Condition"] = df["Condition"].str.title()

   return df

   
transformed_df = transform_weather(data)

print(transformed_df)





    City Country         Condition  Humidity  Temp_C  Feels_Like_C
0  Zocca      IT  Scattered Clouds        77   15.35         14.95


In [9]:
# load to postgres

import pandas as pd
from sqlalchemy import create_engine
from transform import transformed_df
from dotenv import load_dotenv
import os



def load_to_postgres():

    
    load_dotenv(override=True)

    host     = os.getenv("DB_HOST")
    port     = os.getenv("DB_PORT")
    user     = os.getenv("DB_USER")
    dbname   = os.getenv("DB_NAME")
    password = os.getenv("DB_PASSWORD")
    sslmode  = "require"
    
    engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")


    try:
        with engine.begin() as conn:

            transformed_df.to_sql("weather_report", con=conn, if_exists="append", index=False)

            print("\n[SUCCESS] Data successfully pushed to SQL table 'weather_report'")
        
        with engine.connect() as conn:

            result = pd.read_sql("SELECT * FROM weather_report ORDER BY 1 DESC LIMIT 1", conn)

            print("\n--- Recent Rows in Database ---")

            print(result)

        return True
    
    except Exception as e:
        print(f"\n[ERROR] Postgres Load Failed: {e}")
        return False
    
load_to_postgres()






[SUCCESS] Data successfully pushed to SQL table 'weather_report'

--- Recent Rows in Database ---
    City Country      Condition  Temp_C  Feels_Like_C  Humidity
0  Zocca      IT  broken clouds   18.97         18.57        63


True

In [ ]:
# etl

import os
import psycopg2
import requests
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

def run_weather_pipeline():
    # 1. Setup & Environment
    load_dotenv(override=True)
    
    # API Config
    API_KEY = os.getenv("API_KEY")
    LAT = os.getenv("LAT")
    LON = os.getenv("LON")
    
    # DB Config
    host = os.getenv("DB_HOST")
    port = os.getenv("DB_PORT", "5432")
    user = os.getenv("DB_USER")
    dbname = os.getenv("DB_NAME")
    password = os.getenv("DB_PASSWORD")

    print("--- Starting Weather ETL Pipeline ---")

    try:
        # --- STAGE 1: EXTRACT ---
        print("[1/3] Extracting data...")
        url = f"https://api.openweathermap.org/data/2.5/weather?lat={LAT}&lon={LON}&appid={API_KEY}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        raw_data = response.json()

        # --- STAGE 2: TRANSFORM ---
        print("[2/3] Transforming data...")
        # Create dictionary for DataFrame
        weather_data = {
            "City": raw_data.get("name"),
            "Country": raw_data.get("sys", {}).get("country"),
            "Condition": raw_data.get("weather", [{}])[0].get("description", "").title(),
            "Temp_C": round(raw_data.get("main", {}).get("temp", 0) - 273.15, 2),
            "Feels_Like_C": round(raw_data.get("main", {}).get("feels_like", 0) - 273.15, 2),
            "Humidity": raw_data.get("main", {}).get("humidity")
        }
        df = pd.DataFrame([weather_data])
        print(df)

        # --- STAGE 3: LOAD ---
        print(f"[3/3] Connecting to {host}...")
        engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")


        with engine.begin() as conn:
            df.to_sql("weather_report", con=conn, if_exists="append", index=False)
        
        print("\n--- [SUCCESS] ETL Process Finished Successfully ---")

    except Exception as e:
        # This will catch ANY error in Stage 1, 2, or 3
        print(f"\n--- [FAILURE] Pipeline stopped due to error: ---")
        print(f"Error details: {e}")

if __name__ == "__main__":
    run_weather_pipeline()

--- Starting Weather ETL Pipeline ---
[1/3] Extracting data...
[2/3] Transforming data...
    City Country         Condition  Temp_C  Feels_Like_C  Humidity
0  Zocca      IT  Scattered Clouds   16.43         16.17        78
[3/3] Connecting to pg-138e7cb9-damaris-2a4c.d.aivencloud.com...

--- [SUCCESS] ETL Process Finished Successfully ---
